In [1]:
pip install micrograd


Note: you may need to restart the kernel to use updated packages.


In [9]:
#from micrograd.engine import Value

In [11]:
import numpy
import math 
class Value:
    def __init__(self, data, child = () ,_op = " " ):
        self.data = data
        self.grad = 0
        self._backward = lambda:None
        self._prev = set(child)
        self._op = _op
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other ,Value) else Value(other)
        out = Value(self.data + other.data, child = (self, other) ,_op = "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out    
     
    def __mul__(self, other):
        other = other if isinstance(other ,Value) else Value(other)
        out = Value(self.data * other.data, child = (self, other) ,_op = "*")
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def __rmul__(self, other): # other * self
        return self * other
    def __pow__(self ,other):
        assert isinstance(other, (int, float)),"only supporting int/float powers for now"
            
        
        out = Value(self.data ** other ,(self,) ,f"power by {other}")
        def _backward():
            self.grad += (other * self.data ** (other-1)) * out.grad
        out._backward = _backward
        return out     
    def tanh(self):
        x = self.data
        y = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(y , (self , ), "tanh")
        def _backward():
            self.grad += (1 - y**2) * out.grad
            
        out._backward = _backward
        return out
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v) 
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()
    def __sub__(self ,other):
        return self + (-other)
    def __rsub__(self ,other):
        return other + (-self)
    def __radd__(self ,other):
        return self + other
    def __truediv__(self ,other):
        return self * other ** -1
    def __rtruediv__(self ,other):
        return other * self ** -1
    def __neg__(self):
        return self * -1

IndentationError: unindent does not match any outer indentation level (<string>, line 5)